<a href="https://colab.research.google.com/github/xiaoruiyu405/dse-ml-2026/blob/main/processing_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install bertopic
from bertopic.backend import BaseEmbedder
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer


class TfIdfEmbedder(BaseEmbedder):
    def __init__(self, vocabulary: list[str] | None = None, *args, **kwargs):
        super().__init__()
        self.embedding_model: TfidfVectorizer = TfidfVectorizer(*args, **kwargs)
        self.vocabulary = vocabulary
        if vocabulary is not None:
            self.embedding_model.fit(self.vocabulary)

    def embed(self, documents, verbose=True):
        if self.vocabulary is None:
            print("Call 'fit_transform'...")
            return self.embedding_model.fit_transform(documents)
        print("Call 'transform' only...")
        return self.embedding_model.transform(documents)


class CountVectorizerEmbedder(BaseEmbedder):
    def __init__(self, vocabulary: list[str] | None = None, *args, **kwargs):
        super().__init__()
        self.embedding_model: CountVectorizer = CountVectorizer(*args, **kwargs)
        self.vocabulary = vocabulary
        if vocabulary is not None:
            self.embedding_model.fit(self.vocabulary)

    def embed(self, documents, verbose=True):
        if self.vocabulary is None:
            print("Call 'fit_transform'...")
            return self.embedding_model.fit_transform(documents)
        print("Call 'transform' only...")
        return self.embedding_model.transform(documents)


class SentenceTransformerSmall(BaseEmbedder):
    def __init__(self):
        super().__init__()
        self.embedding_model = SentenceTransformer("all-minilm-l6-v2")

    def embed(self, documents, verbose=True):
        embeddings = self.embedding_model.encode(documents, show_progress_bar=verbose)
        return embeddings


class SentenceTransformerLarge(BaseEmbedder):
    def __init__(self):
        super().__init__()
        self.embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

    def embed(self, documents, verbose=True):
        embeddings = self.embedding_model.encode(documents, show_progress_bar=verbose)
        return embeddings

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 9.9 MB/s eta 0:00:00


In [7]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

tqdm.pandas()

### 1. Prepare parlamint dataset

#### 1.1. Load Dataset (Sentence-Wise)

In [12]:
# Load parlamint dataset
df_parlamint = pd.read_csv("/content/drive/MyDrive/dse-ml-2026/materials/datasets/parlamint/parlamint-it-is-2022.txt", sep="\t").head(10000)
df_parlamint_subset = df_parlamint.copy(deep=True).head(100)
df_parlamint

,ID,Parent_ID,Text
0,ParlaMint-IS_2022-01-17-20.seg2.1,ParlaMint-IS_2022-01-17-20.u1,President of the United States reports:
1,ParlaMint-IS_2022-01-17-20.seg3.1,ParlaMint-IS_2022-01-17-20.u1,"I have decided, according to the proposal of t..."
2,ParlaMint-IS_2022-01-17-20.seg4.1,ParlaMint-IS_2022-01-17-20.u1,"Arrange sites, January 11th, 2022."
3,ParlaMint-IS_2022-01-17-20.seg6.1,ParlaMint-IS_2022-01-17-20.u1,Katrín Jakobsdóttir's daughter.
4,ParlaMint-IS_2022-01-17-20.seg7.1,ParlaMint-IS_2022-01-17-20.u1,Presidential Letters for a meeting of the Gene...
...,...,...,...
9995,ParlaMint-IS_2022-01-27-28.seg113.4,ParlaMint-IS_2022-01-27-28.u58,"Some of these are good, but the big picture is..."
9996,ParlaMint-IS_2022-01-27-28.seg113.5,ParlaMint-IS_2022-01-27-28.u58,The variable was never taken out whether it wa...
9997,ParlaMint-IS_2022-01-27-28.seg114.1,ParlaMint-IS_2022-01-27-28.u58,I will not vote with this case.
9998,ParlaMint-IS_2022-01-27-28.seg114.2,ParlaMint-IS_2022-01-27-28.u58,I won't get in the way of this case.


In [11]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#### 1.2. Group Dataset per Utterance

In [13]:
# Group sentence by utterance (=Parent_ID)
df_parlamint_grouped = (df_parlamint.groupby(["Parent_ID"])["Text"]
                        .apply(lambda s: " ".join(s))
                        .reset_index(name="utterance_text"))
print(f"Unique utterances: {df_parlamint_grouped.shape[0]}")
df_parlamint_grouped

Unique utterances: 660


,Parent_ID,utterance_text
0,ParlaMint-IS_2022-01-17-20.u1,President of the United States reports: I have...
1,ParlaMint-IS_2022-01-17-20.u10,"Before the weekend, an article by Stefánssonar..."
2,ParlaMint-IS_2022-01-17-20.u11,"I read this decision in Perconte, which is not..."
3,ParlaMint-IS_2022-01-17-20.u12,"In fact, this is shown in the letter quoted by..."
4,ParlaMint-IS_2022-01-17-20.u13,"Yes, that's right. That's right. A senator who..."
...,...,...
655,ParlaMint-IS_2022-01-27-28.u58,Here we vote for a case that involves massive ...
656,ParlaMint-IS_2022-01-27-28.u6,"I come up here to agree with this, this case i..."
657,ParlaMint-IS_2022-01-27-28.u7,In his article at Science yesterday and also i...
658,ParlaMint-IS_2022-01-27-28.u8,The president still reminds us of a limited ta...


In [14]:
sample_utterance = df_parlamint[df_parlamint["Parent_ID"] == "ParlaMint-IS_2022-01-17-20.u1"]["Text"]
sample_utterance

,Text
0,President of the United States reports:
1,"I have decided, according to the proposal of t..."
2,"Arrange sites, January 11th, 2022."
3,Katrín Jakobsdóttir's daughter.
4,Presidential Letters for a meeting of the Gene...
5,I'd like to use this opportunity here after re...


### 2. Different Text Embedding Algorithms

#### 2.1. Count Vectorizer (Sparse) Embeddings

In [15]:
# Adding the whole parlamint dataset as vocabulary
# cv_model = CountVectorizerEmbedder(vocabulary=df_parlamint["Text"].to_list(), min_df=100, stop_words='english',
#                                    n_gram_range=(1, 3))

# Adding just the utterance sample as vocabulary
cv_model = CountVectorizerEmbedder(vocabulary=df_parlamint_grouped["utterance_text"], max_features=10,stop_words='english')

In [ ]:
cv_embeddings = cv_model.embed(sample_utterance)
print(f"Number features: {len(cv_model.embedding_model.get_feature_names_out())}", cv_model.embedding_model.get_feature_names_out())
print(f"Shape embedding array: {cv_embeddings.toarray().shape}")
df_cv_output = pd.DataFrame(columns=cv_model.embedding_model.get_feature_names_out(), data=cv_embeddings.toarray())
df_cv_output

Call 'transform' only...
Number features: 10 ['council' 'government' 'health' 'just' 'like' 'minister' 'need' 'people'
 'think' 'time']
Shape embedding array: (6, 10)


,council,government,health,just,like,minister,need,people,think,time
0,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,1,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,1,0,0,0,0,0


#### 2.2 TF-IDF (Sparse) Embeddings

In [16]:
# Adding the whole parlamint dataset as vocabulary
# tfidf_model = TfIdfEmbedder(vocabulary=df_parlamint["Text"].to_list(), min_df=100, stop_words='english')

# Adding just the utterance sample as vocabulary
tfidf_model = TfIdfEmbedder(vocabulary=df_parlamint_grouped["utterance_text"], max_features=10, stop_words='english')

In [ ]:
tfidf_embeddings = tfidf_model.embed(sample_utterance)
print(f"Number features: {len(tfidf_model.embedding_model.get_feature_names_out())}", tfidf_model.embedding_model.get_feature_names_out())
print(f"Shape embedding array: {tfidf_embeddings.toarray().shape}")
df_tfidf_output = pd.DataFrame(columns=tfidf_model.embedding_model.get_feature_names_out(), data=tfidf_embeddings.toarray())
df_tfidf_output

Call 'transform' only...
Number features: 10 ['council' 'government' 'health' 'just' 'like' 'minister' 'need' 'people'
 'think' 'time']
Shape embedding array: (6, 10)


,council,government,health,just,like,minister,need,people,think,time
0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
1,0.735172,0.0,0.0,0.0,0.0,0.677881,0.0,0.0,0.0,0.0
2,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
3,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
4,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
5,0.000000,0.0,0.0,0.0,1.0,0.000000,0.0,0.0,0.0,0.0


#### 2.3 Sentence Transformer (Dense) Embeddings

In [17]:
st_model_small = SentenceTransformer('all-minilm-l6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [18]:
# Encode sentence-wise
# sample_utterance is a pandas Series; encode expects a string or a list of strings.
st_embeddings = st_model_small.encode(sample_utterance.tolist())
print(f"Number of sentence embeddings: {len(st_embeddings)}")
print(f"Shape embedding array: {st_embeddings.shape}")
st_embeddings

Number of sentence embeddings: 6
Shape embedding array: (6, 384)


array([[ 0.00364315,  0.0075753 , -0.01352419, ..., -0.01752402,
         0.00990044,  0.06021778],
       [-0.04760474, -0.06898868,  0.02710493, ..., -0.06230193,
        -0.14385448,  0.03971656],
       [-0.01698016, -0.04804511, -0.01744974, ..., -0.03905804,
        -0.07595797,  0.00352384],
       [-0.08331395, -0.04858721,  0.00143909, ...,  0.03839727,
         0.06088394, -0.01607143],
       [-0.07626822, -0.06527785,  0.06610004, ...,  0.02295852,
        -0.10826011, -0.06591922],
       [ 0.01009847, -0.0453703 ,  0.04564784, ..., -0.00065686,
        -0.0854706 , -0.00425592]], dtype=float32)

In [19]:
# Encode utterance-wise
st_embeddings_u = st_model_small.encode(" ".join(sample_utterance))
print(f"Number features: {len(st_embeddings_u)}")
print(f"Shape embedding array: {st_embeddings_u.shape}")
st_embeddings_u

Number features: 384
Shape embedding array: (384,)


array([-5.19020185e-02, -1.02959432e-01,  6.31395429e-02,  2.98001897e-02,
       -2.51417439e-02, -9.66061372e-03, -7.38494396e-02, -2.23561283e-02,
       -6.58514723e-02,  7.95470644e-03, -6.14717044e-02,  9.11198277e-03,
       -7.41380826e-02, -8.91720783e-03,  4.29924875e-02,  3.98151763e-02,
        5.71963436e-04, -2.14750301e-02,  4.03534435e-02, -3.10852192e-03,
        4.81391363e-02,  3.13877873e-02,  2.43081450e-02,  3.66791897e-03,
       -4.10232022e-02, -1.00539709e-02, -1.73179135e-02, -3.86848114e-02,
       -1.10482778e-02,  6.46823421e-02,  5.51152155e-02, -5.29571762e-03,
        7.86908716e-02,  2.21064594e-02,  5.85365146e-02,  2.10544281e-03,
        5.57183810e-02,  3.54421027e-02,  3.88754159e-02, -5.69796637e-02,
       -1.24397287e-02, -6.79902136e-02,  2.59145629e-02, -4.62712254e-03,
       -3.93818915e-02,  4.40331772e-02, -3.18673030e-02, -2.18536193e-03,
       -3.43883298e-02,  6.45774975e-02,  4.68759844e-03,  1.60151464e-03,
        1.24297533e-02, -

### 3. Encode whole Parlamint Dataset

#### 3.1 Encode with Sentence Transformer

In [20]:
# Encode utterance-wise dataset
df_parlamint_embeddings_per_utterance = st_model_small.encode(df_parlamint_grouped["utterance_text"].to_list(),
                                                     show_progress_bar=True)

# Encode sentence-wise dataset
df_parlamint_embeddings_per_sentence = st_model_small.encode(df_parlamint["Text"].to_list(), show_progress_bar=True)

Batches:   0%|          | 0/21 [00:00<?, ?it/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

In [21]:
df_parlamint_grouped["embedding"] = list(df_parlamint_embeddings_per_utterance)
df_parlamint_grouped

,Parent_ID,utterance_text,embedding
0,ParlaMint-IS_2022-01-17-20.u1,President of the United States reports: I have...,"[-0.05190201, -0.102959484, 0.063139565, 0.029..."
1,ParlaMint-IS_2022-01-17-20.u10,"Before the weekend, an article by Stefánssonar...","[-0.104611516, 0.06670193, -0.056569587, -0.02..."
2,ParlaMint-IS_2022-01-17-20.u11,"I read this decision in Perconte, which is not...","[-0.0037422264, 0.07299824, -0.017051956, -0.0..."
3,ParlaMint-IS_2022-01-17-20.u12,"In fact, this is shown in the letter quoted by...","[-0.1063038, 0.06314155, -0.014823508, 0.01412..."
4,ParlaMint-IS_2022-01-17-20.u13,"Yes, that's right. That's right. A senator who...","[-0.03764075, 0.10380772, -0.061545722, 0.0093..."
...,...,...,...
655,ParlaMint-IS_2022-01-27-28.u58,Here we vote for a case that involves massive ...,"[-0.016700177, -0.07529929, 0.029732978, 0.003..."
656,ParlaMint-IS_2022-01-27-28.u6,"I come up here to agree with this, this case i...","[-0.0046804934, 0.05790165, -0.003511662, -0.0..."
657,ParlaMint-IS_2022-01-27-28.u7,In his article at Science yesterday and also i...,"[-0.026387008, 0.05542075, 0.045418274, 0.0204..."
658,ParlaMint-IS_2022-01-27-28.u8,The president still reminds us of a limited ta...,"[0.017418494, -0.035377957, 0.09752447, -0.021..."


In [22]:
df_parlamint["embedding"] = list(df_parlamint_embeddings_per_sentence)
df_parlamint

,ID,Parent_ID,Text,embedding
0,ParlaMint-IS_2022-01-17-20.seg2.1,ParlaMint-IS_2022-01-17-20.u1,President of the United States reports:,"[0.0036431411, 0.0075753005, -0.013524175, 0.0..."
1,ParlaMint-IS_2022-01-17-20.seg3.1,ParlaMint-IS_2022-01-17-20.u1,"I have decided, according to the proposal of t...","[-0.047604747, -0.06898866, 0.027104955, 0.042..."
2,ParlaMint-IS_2022-01-17-20.seg4.1,ParlaMint-IS_2022-01-17-20.u1,"Arrange sites, January 11th, 2022.","[-0.016980128, -0.04804515, -0.017449748, 0.01..."
3,ParlaMint-IS_2022-01-17-20.seg6.1,ParlaMint-IS_2022-01-17-20.u1,Katrín Jakobsdóttir's daughter.,"[-0.08331394, -0.04858722, 0.0014390874, -0.04..."
4,ParlaMint-IS_2022-01-17-20.seg7.1,ParlaMint-IS_2022-01-17-20.u1,Presidential Letters for a meeting of the Gene...,"[-0.07626822, -0.06527785, 0.06610004, 0.01071..."
...,...,...,...,...
9995,ParlaMint-IS_2022-01-27-28.seg113.4,ParlaMint-IS_2022-01-27-28.u58,"Some of these are good, but the big picture is...","[-0.010372346, -0.04594692, -0.015732536, -0.0..."
9996,ParlaMint-IS_2022-01-27-28.seg113.5,ParlaMint-IS_2022-01-27-28.u58,The variable was never taken out whether it wa...,"[0.0029584889, 0.07198218, -0.037595496, 0.068..."
9997,ParlaMint-IS_2022-01-27-28.seg114.1,ParlaMint-IS_2022-01-27-28.u58,I will not vote with this case.,"[-0.0003005359, 0.038990907, 0.017069632, -0.0..."
9998,ParlaMint-IS_2022-01-27-28.seg114.2,ParlaMint-IS_2022-01-27-28.u58,I won't get in the way of this case.,"[-0.035732668, 0.06427993, 0.012762982, 0.0055..."


#### 3.2 Save output to pickle file

In [23]:
df_parlamint.to_pickle("df_parlamint_all-MiniLM-L6-v2.pkl")

#### 3.3 Encode Dataset with TF-IDF

In [24]:
# Adding the whole parlamint dataset as vocabulary
tfidf_model = TfIdfEmbedder(vocabulary=df_parlamint["Text"].to_list(), max_features=100, stop_words='english')

# Encode sentence-wise dataset
tfidf_embeddings_per_sentence = tfidf_model.embed(df_parlamint["Text"].to_list())

Call 'transform' only...


In [25]:
print(f"Number features: {len(tfidf_model.embedding_model.get_feature_names_out())}", tfidf_model.embedding_model.get_feature_names_out())
print(f"Shape embedding array: {tfidf_embeddings_per_sentence.toarray().shape}")
tfidf_embeddings_per_sentence.toarray()

Number features: 100 ['000' 'able' 'agree' 'ask' 'believe' 'better' 'business' 'care' 'case'
 'change' 'changes' 'children' 'clear' 'come' 'committee' 'community'
 'companies' 'council' 'country' 'course' 'decision' 'discussion' 'does'
 'don' 'economic' 'epidemic' 'especially' 'example' 'fact' 'financial'
 'general' 'going' 'good' 'government' 'health' 'highest' 'hospital'
 'iceland' 'important' 'increase' 'just' 'know' 'land' 'law' 'laws' 'like'
 'long' 'look' 'lot' 'make' 'management' 'matter' 'measures' 'members'
 'mental' 'minister' 'ministers' 'ministry' 'money' 'national' 'need'
 'new' 'number' 'order' 'pay' 'people' 'place' 'point' 'possible'
 'president' 'problem' 'public' 'really' 'report' 'right' 'rights' 'said'
 'say' 'security' 'senator' 'service' 'situation' 'social' 'society'
 'state' 'support' 'taken' 'talk' 'thank' 'things' 'think' 'time' 'today'
 'use' 've' 'want' 'way' 'work' 'year' 'years']
Shape embedding array: (10000, 100)


array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [26]:
df_parlamint["embedding"] = list(tfidf_embeddings_per_sentence.toarray())
df_parlamint

,ID,Parent_ID,Text,embedding
0,ParlaMint-IS_2022-01-17-20.seg2.1,ParlaMint-IS_2022-01-17-20.u1,President of the United States reports:,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,ParlaMint-IS_2022-01-17-20.seg3.1,ParlaMint-IS_2022-01-17-20.u1,"I have decided, according to the proposal of t...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,ParlaMint-IS_2022-01-17-20.seg4.1,ParlaMint-IS_2022-01-17-20.u1,"Arrange sites, January 11th, 2022.","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,ParlaMint-IS_2022-01-17-20.seg6.1,ParlaMint-IS_2022-01-17-20.u1,Katrín Jakobsdóttir's daughter.,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,ParlaMint-IS_2022-01-17-20.seg7.1,ParlaMint-IS_2022-01-17-20.u1,Presidential Letters for a meeting of the Gene...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
...,...,...,...,...
9995,ParlaMint-IS_2022-01-27-28.seg113.4,ParlaMint-IS_2022-01-27-28.u58,"Some of these are good, but the big picture is...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
9996,ParlaMint-IS_2022-01-27-28.seg113.5,ParlaMint-IS_2022-01-27-28.u58,The variable was never taken out whether it wa...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
9997,ParlaMint-IS_2022-01-27-28.seg114.1,ParlaMint-IS_2022-01-27-28.u58,I will not vote with this case.,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, ..."
9998,ParlaMint-IS_2022-01-27-28.seg114.2,ParlaMint-IS_2022-01-27-28.u58,I won't get in the way of this case.,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.719..."


#### 3.4 Save output to pickle file

In [27]:
df_parlamint.to_pickle("df_parlamint_all-tfidf.pkl")

#### 3.5 Load data from pickle file

In [28]:
# df_read_parlamint = pd.read_pickle("<filename_path>.pkl")
df_parlamint = pd.read_pickle("df_parlamint_all-MiniLM-L6-v2.pkl")

### 4. Calculate similarities between embeddings

In [29]:
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics.pairwise import cosine_similarity

st_model_small = SentenceTransformer('all-minilm-l6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [30]:
# 1. Example data
sentences = [
    "I deposited my paycheck at the bank yesterday.",
    "We had a picnic on the bank of the river.",
    "The financial institution announced a new savings account plan.",
    "She withdrew cash from the nearest ATM.",
    "The kids played near the riverbank after school."
]

query = "financial services"

# 2. Sentence Transformers embeddings
dense_embeddings_sentences = st_model_small.encode(sentences, convert_to_tensor=False)
dense_embeddings_query = st_model_small.encode([query], convert_to_tensor=False)
dense_similarities = util.cos_sim(dense_embeddings_query, dense_embeddings_sentences)[0].cpu().numpy()

# 3. TF-IDF embeddings
tfidf_model = TfIdfEmbedder(vocabulary=sentences, max_features=10, stop_words='english')
tfidf_embeddings_sentences = tfidf_model.embed(sentences)
tfidf_embeddings_query = tfidf_model.embed([query])
tfidf_similarities = cosine_similarity(tfidf_embeddings_query, tfidf_embeddings_sentences).flatten()

# 4. Compare rankings
df = pd.DataFrame({
    "sentence": sentences,
    "tfidf_similarity": tfidf_similarities,
    "st_similarity": dense_similarities
})
# df.sort_values(by=["st_similarity"], ascending=False, inplace=True)
df

Call 'transform' only...
Call 'transform' only...


,sentence,tfidf_similarity,st_similarity
0,I deposited my paycheck at the bank yesterday.,0.0,0.243795
1,We had a picnic on the bank of the river.,0.0,0.047487
2,The financial institution announced a new savi...,0.5,0.322175
3,She withdrew cash from the nearest ATM.,0.0,0.237477
4,The kids played near the riverbank after school.,0.0,0.080024


### 5. How to build a Simple QA System

#### 5.1 Get the Most Likely Utterance

In [31]:
import numpy as np
from sentence_transformers import util

# Given question
question = "What is the government policy on climate change?"
# question = "What about president of america?"

# 1. Embed the question
question_embedding = st_model_small.encode(question)

# 2. Compute cosine similarities
cosine_similarities = util.cos_sim(question_embedding, df_parlamint["embedding"])[0].cpu().numpy()

# 3. Get the index of the most similar utterance
most_similar_idx = int(np.argmax(cosine_similarities))

# 4. Retrieve the most similar text
most_similar_text = df_parlamint.iloc[most_similar_idx]["Text"]
# most_similar_text
print(f"Score: {cosine_similarities[most_similar_idx]:.4f} | Utterance: {most_similar_text}\n")

Score: 0.6772 | Utterance: So we're going to make it even. by the Minister's implementation of the government's strategy for climate change and by the process of climate management in its new formulation, public analysis of measures and other government policies.



In [32]:
len(df_parlamint["embedding"].iloc[1])

384

In [33]:
df_parlamint["Text"]

,Text
0,President of the United States reports:
1,"I have decided, according to the proposal of t..."
2,"Arrange sites, January 11th, 2022."
3,Katrín Jakobsdóttir's daughter.
4,Presidential Letters for a meeting of the Gene...
...,...
9995,"Some of these are good, but the big picture is..."
9996,The variable was never taken out whether it wa...
9997,I will not vote with this case.
9998,I won't get in the way of this case.


#### 5.2 Get the Top-K relevant Utterances

In [34]:
question = "What is the government policy on climate change?"
# question = "America?"
k = 5  # choose how many results you want

# 1. Embed the question
question_embedding = st_model_small.encode(question)

# 2. Compute cosine similarities
cosine_similarities = util.cos_sim(question_embedding, df_parlamint["embedding"])[0].cpu().numpy()

# 3. Get indices of top-k most similar utterances
top_k_idx = np.argsort(cosine_similarities)[::-1][:k]

# 4. Retrieve the top-k utterances and their similarity scores
for idx in top_k_idx:
    text = df_parlamint.iloc[idx]["Text"]
    score = cosine_similarities[idx]
    print(f"Score: {score:.4f} | Utterance: {text}\n")


Score: 0.6772 | Utterance: So we're going to make it even. by the Minister's implementation of the government's strategy for climate change and by the process of climate management in its new formulation, public analysis of measures and other government policies.

Score: 0.6287 | Utterance: Is it consistent with the left-green climate policy?

Score: 0.6216 | Utterance: Does Ministers believe that a silicar's restarting can harmonize with the outlook and policy of the nation on climate?

Score: 0.6040 | Utterance: There is an emergency, Mrs. President, on climate issues, and this bill responds to that crisis by suggesting changes that will enable governments to be more effective and that will simply require them to face the climate crisis, the most important solution we are facing.

Score: 0.5818 | Utterance: In the 3rd. The bill suggests that the government's action programme on climate matters will be much more extensive, a better estimate of the estimated costs and the assessment of

In [35]:
tfidf_model = TfIdfEmbedder(vocabulary=df_parlamint["Text"], max_features=1000, stop_words='english')
tfidf_embeddings_sentences = tfidf_model.embed(df_parlamint["Text"].to_list())

Call 'transform' only...


In [36]:
df_parlamint["embedding"] = list(tfidf_embeddings_sentences.toarray())

In [37]:
question = "What is the government policy on climate change?"
# question = "America?"
k = 5  # choose how many results you want

# 1. Embed the question
#question_embedding = tfidf_model.encode(question)
question_embedding = tfidf_model.embed([question])

# 2. Compute cosine similarities
cosine_similarities = util.cos_sim(question_embedding.toarray(), df_parlamint["embedding"])[0].cpu().numpy()

# 3. Get indices of top-k most similar utterances
top_k_idx = np.argsort(cosine_similarities)[::-1][:k]

# 4. Retrieve the top-k utterances and their similarity scores
for idx in top_k_idx:
    text = df_parlamint.iloc[idx]["Text"]
    score = cosine_similarities[idx]
    print(f"Score: {score:.4f} | Utterance: {text}\n")


Call 'transform' only...
Score: 0.6444 | Utterance: Synchronize bill with government policy.

Score: 0.5579 | Utterance: Climate change is here and it's today, just like us.

Score: 0.5479 | Utterance: The government must be stopped after a health-irritation policy.

Score: 0.5448 | Utterance: There is no regional policy in this.

Score: 0.5416 | Utterance: Is it consistent with the left-green climate policy?



In [38]:
import numpy as np
np.where(question_embedding.toarray() > 0)


(array([0, 0, 0, 0]), array([134, 148, 385, 674]))

In [41]:
print(f"Number features: {len(tfidf_model.embedding_model.get_feature_names_out())}", tfidf_model.embedding_model.get_feature_names_out())
print(f"Shape embedding array: {tfidf_embeddings_sentences.toarray().shape}")
df_tfidf_output = pd.DataFrame(columns=tfidf_model.embedding_model.get_feature_names_out(), data=tfidf_embeddings_sentences.toarray())

Number features: 1000 ['000' '10' '12' '15' '17' '18' '19' '1st' '20' '2018' '2020' '2021'
 '2022' '30' '40' '50' '500' 'ability' 'able' 'abroad' 'absolutely'
 'abuse' 'accept' 'accepted' 'access' 'accordance' 'according' 'account'
 'achieve' 'act' 'action' 'actions' 'activation' 'active' 'activities'
 'activity' 'actually' 'add' 'added' 'addiction' 'addition' 'additional'
 'adjustments' 'administration' 'administrative' 'adopted' 'advance'
 'advantage' 'advice' 'advisers' 'affairs' 'affect' 'affected' 'afraid'
 'age' 'agency' 'ago' 'agree' 'agreed' 'agreement' 'agreements' 'ahead'
 'air' 'airport' 'allow' 'allowed' 'analysis' 'answer' 'answers'
 'apartments' 'application' 'applications' 'applies' 'apply' 'approach'
 'appropriate' 'area' 'areas' 'arrangement' 'article' 'ask' 'asked'
 'asking' 'assembly' 'assessment' 'attention' 'authorities' 'authority'
 'available' 'average' 'aware' 'away' 'bad' 'balance' 'bank' 'based'
 'basic' 'basis' 'began' 'beginning' 'believe' 'benefit' 'benefit